In [ ]:
import numpy as np

SEED_DIGIT = 6               
rng = np.random.default_rng(SEED_DIGIT)

X = np.genfromtxt("secom.data")
n_rows, n_feat = X.shape

miss_pct_all = np.isnan(X).mean(axis=0) * 100
with np.errstate(all="ignore"):
    constant = (np.nanmax(X, axis=0) == np.nanmin(X, axis=0))
all_missing = miss_pct_all == 100
usable = np.flatnonzero(~constant & ~all_missing)

k = 5 + (SEED_DIGIT % 3)
sel = np.sort(rng.choice(usable, size=k, replace=False))
S = X[:, sel]

means   = np.nanmean(S, axis=0)
medians = np.nanmedian(S, axis=0)
stds    = np.nanstd(S, axis=0, ddof=1)      
miss    = np.isnan(S).mean(axis=0) * 100

mad = np.nanmedian(np.abs(S - medians), axis=0)
safe_mad = np.where(mad == 0, np.nan, mad)
rz = 0.6745 * (S - medians) / safe_mad
rz = np.where(np.isnan(rz), 0.0, rz)         

flags = np.abs(rz) > 3.5
n_obs = (~np.isnan(S)).sum(axis=0)
flag_pct = flags.sum(axis=0) / n_obs * 100

order = np.argsort(-flag_pct)
header = "rank,feature_index,mean,median,std,missing_pct,flagged_count,flagged_pct"
rows = np.column_stack([
    np.arange(1, k + 1), sel[order], means[order], medians[order],
    stds[order], miss[order], flags.sum(axis=0)[order], flag_pct[order]
])
np.savetxt("measurement_quality.csv", rows, delimiter=",", header=header,
           comments="", fmt=["%d", "%d"] + ["%.4f"] * 4 + ["%d", "%.4f"])

print("Selected Features:", sel.tolist())
print("Highest Missing Percentage:", round(miss_pct_all.max(), 2), "(all features)",
      "| among selected:", round(miss.max(), 2))
print("Feature with Highest Flagged %:", sel[order[0]], round(flag_pct[order[0]], 2))
print("Total Flagged Observations:", int(flags.sum()))

ok = True
for j in range(k):
    col = S[:, j][~np.isnan(S[:, j])]
    med = np.median(col); m = np.median(np.abs(col - med))
    cnt = 0 if m == 0 else int(np.sum(np.abs(0.6745 * (col - med) / m) > 3.5))
    ok &= (cnt == flags[:, j].sum())
print("Vectorized Results Verified:", "Yes" if ok else "No")

Selected Features: [171, 228, 289, 298, 564]
Highest Missing Percentage: 91.19 (all features) | among selected: 17.42
Feature with Highest Flagged %: 564 8.42
Total Flagged Observations: 260
Vectorized Results Verified: Yes
